# Traffic Congestion Prediction: MobileNet + Echo State Network (ESN)

This notebook implements a **training-free** pipeline for video classification, suitable for cloud environments (Kaggle/Colab).

### Approach (Option 4):
1.  **Spatial Features**: Use a **Frozen Pre-trained MobileNetV2** to extract high-level visual features (1280-dim) from each frame. No backpropagation is performed on the CNN.
2.  **Temporal Features**: Use an **Echo State Network (ESN)** (Reservoir Computing) to model the temporal evolution of these features over time.
3.  **Classification**: Train a linear readout (Ridge Regression) to predict traffic congestion levels.

**Advantages**: 
- **Extremely Fast Training**: No gradient descent, just one-shot matrix solution.
- **Low Compute**: Can perform reasonably well without high-end GPUs for training.
- **Video-Native**: Processes raw video frames.

In [1]:
# Install dependencies
!pip install --upgrade numpy keras tensorflow opencv-python-headless scikit-learn polars

## 1. Configuration & Paths
Please set the following paths to your dataset locations.

In [9]:
import os
import cv2
import numpy as np
import pandas as pd
import polars as pl
import tensorflow
import tensorflow as tf
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
from sklearn.linear_model import Ridge
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import train_test_split
from google.cloud import storage

# --- USER CONFIGURATION ---
BASE_DIR = '/teamspace/studios/this_studio/Barbados_Traffic_Analysis_Challenge_dev' # Example
VIDEO_DIR = '/teamspace/studios/this_studio/videos' # Example
TRAIN_CSV = os.path.join(BASE_DIR, 'demos/Train_Balanced_3k.csv')
TEST_CSV = os.path.join(BASE_DIR, 'demos/TestInputSegments.csv')
SAMPLE_SUB = os.path.join(BASE_DIR, 'demos/SampleSubmission.csv')

# Model Config
IMG_SIZE = (224, 224)
SEQ_LENGTH = 30         # Number of frames to extract per video (downsampled)
RESERVOIR_DIM = 1000    # ESN Reservoir Size
SPECTRAL_RADIUS = 0.9
LEAK_RATE = 0.2
RIDGE_ALPHA = 1.0
# --------------------------

In [3]:
from google.oauth2 import service_account

# If you uploaded the file to a dataset:
os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = "/teamspace/studios/this_studio/tokens.json"


In [4]:
client = storage.Client(project="brb-traffic")

# Base directories and bucket name
bucket_name = 'brb-traffic'

# Video paths
video_dir = VIDEO_DIR
video_path = '/teamspace/studios/this_studio/videos'
os.makedirs(video_dir, exist_ok=True)

# Datasheet paths
train_csv_path = TRAIN_CSV
sample_submission_csv_path = SAMPLE_SUB

In [5]:
# Load the dataset
train = pd.read_csv(TRAIN_CSV)

# Extract camera ID and reconstruct the video path
def extract_camera_path(video):
    parts = video.split('/')
    filename = parts[-1]
    camera_id = filename.split('_')[0]
    return f"{camera_id}/{filename}"

train['videos'] = train['videos'].apply(extract_camera_path)

# Display shape and preview
display(train.shape, train.head())

(3000, 14)

,responseId,view_label,ID_enter,ID_exit,videos,video_time,datetimestamp_start,datetimestamp_end,date,signaling,congestion_enter_rating,congestion_exit_rating,time_segment_id,cycle_phase
0,zYkHaeOdB7XOnvgP3YW5kQs,Norman Niles #1,time_segment_0_Norman Niles #1_congestion_ente...,time_segment_0_Norman Niles #1_congestion_exit...,normanniles1/normanniles1_2025-10-20-06-00-45.mp4,2025-10-20 06:00:45,2025-10-20 06:00:45,2025-10-20 06:01:44,2025-10-20,none,free flowing,free flowing,0,train
1,NYsHaeCRLq-vnvgPjoXZqA0,Norman Niles #1,time_segment_1_Norman Niles #1_congestion_ente...,time_segment_1_Norman Niles #1_congestion_exit...,normanniles1/normanniles1_2025-10-20-06-01-45.mp4,2025-10-20 06:01:45,2025-10-20 06:01:45,2025-10-20 06:02:44,2025-10-20,none,free flowing,free flowing,1,train
2,A40HaYT8KNm7nvgPq8e12AU,Norman Niles #1,time_segment_2_Norman Niles #1_congestion_ente...,time_segment_2_Norman Niles #1_congestion_exit...,normanniles1/normanniles1_2025-10-20-06-02-45.mp4,2025-10-20 06:02:45,2025-10-20 06:02:45,2025-10-20 06:03:00,2025-10-20,none,free flowing,free flowing,2,train
3,EIsHaanDMK-vnvgPjoXZqA0,Norman Niles #1,time_segment_3_Norman Niles #1_congestion_ente...,time_segment_3_Norman Niles #1_congestion_exit...,normanniles1/normanniles1_2025-10-20-06-03-45.mp4,2025-10-20 06:03:45,2025-10-20 06:03:45,2025-10-20 06:04:44,2025-10-20,none,free flowing,free flowing,3,train
4,RYsHafSeMaqpmecP5vCV0AQ,Norman Niles #1,time_segment_4_Norman Niles #1_congestion_ente...,time_segment_4_Norman Niles #1_congestion_exit...,normanniles1/normanniles1_2025-10-20-06-04-45.mp4,2025-10-20 06:04:45,2025-10-20 06:04:45,2025-10-20 06:04:59,2025-10-20,none,free flowing,free flowing,4,train


In [6]:
ss = pd.read_csv(SAMPLE_SUB)
display(ss.shape,ss.head())

(880, 3)

,ID,Target,Target_Accuracy
0,time_segment_129_Norman Niles #1_congestion_en...,free flowing,free flowing
1,time_segment_130_Norman Niles #1_congestion_en...,heavy delay,heavy delay
2,time_segment_131_Norman Niles #1_congestion_en...,free flowing,free flowing
3,time_segment_132_Norman Niles #1_congestion_en...,heavy delay,heavy delay
4,time_segment_133_Norman Niles #1_congestion_en...,free flowing,free flowing


In [7]:
# %%capture
blobs=train.videos.tolist()[0:2000]
print(f"Number of blobs selected: {len(blobs)}")
display(blobs)

Number of blobs selected: 2000


['normanniles1/normanniles1_2025-10-20-06-00-45.mp4',
 'normanniles1/normanniles1_2025-10-20-06-01-45.mp4',
 'normanniles1/normanniles1_2025-10-20-06-02-45.mp4',
 'normanniles1/normanniles1_2025-10-20-06-03-45.mp4',
 'normanniles1/normanniles1_2025-10-20-06-04-45.mp4',
 'normanniles1/normanniles1_2025-10-20-06-05-45.mp4',
 'normanniles1/normanniles1_2025-10-20-06-06-45.mp4',
 'normanniles1/normanniles1_2025-10-20-06-07-45.mp4',
 'normanniles1/normanniles1_2025-10-20-06-08-45.mp4',
 'normanniles1/normanniles1_2025-10-20-06-09-45.mp4',
 'normanniles1/normanniles1_2025-10-20-06-10-45.mp4',
 'normanniles1/normanniles1_2025-10-20-06-11-45.mp4',
 'normanniles1/normanniles1_2025-10-20-06-12-45.mp4',
 'normanniles1/normanniles1_2025-10-20-06-13-45.mp4',
 'normanniles1/normanniles1_2025-10-20-06-14-45.mp4',
 'normanniles1/normanniles1_2025-10-20-06-15-45.mp4',
 'normanniles1/normanniles1_2025-10-20-06-16-45.mp4',
 'normanniles1/normanniles1_2025-10-20-06-17-45.mp4',
 'normanniles1/normanniles1_

In [ ]:
# %%capture
from google.api_core.exceptions import NotFound

print(f"--- Debugging Blobs Variable ---")
print(f"Type of 'blobs' before loop: {type(blobs)}")
print(f"Content of 'blobs' (first 5): {blobs[:5]}")
print(f"----------------------------------")

# --- Existing loop for downloading files from the 'blobs' list ---
for blob_name in blobs:
    # Ensure blob_name is a string before proceeding
    if not isinstance(blob_name, str):
        print(f"❌ Error: Expected string for blob_name, but got {type(blob_name)}. Skipping.")
        continue

    blob = client.bucket(bucket_name).blob(blob_name)
    file_name = os.path.basename(blob_name)  # get last part after '/'
    local_path = os.path.join(video_path, file_name)

    print(f"Attempting to download blob: '{blob_name}' to '{local_path}'") # Added print for clarity
    try:
        blob.download_to_filename(local_path)
        print("✅ Downloaded to:", local_path)
    except NotFound:
        print(f"❌ Error: '{blob_name}' not found in bucket '{bucket_name}'. Skipping.")
    except Exception as e:
        print(f"❌ An unexpected error occurred while downloading '{blob_name}': {e}")

## 2. Feature Extraction (MobileNetV2)
We load a MobileNetV2 pre-trained on ImageNet, remove the top classification layer, and use global average pooling to get a 1280-dimensional vector for each frame.

In [10]:
def build_feature_extractor():
    base_model = MobileNetV2(
        weights='imagenet', 
        include_top=False, 
        pooling='avg',
        input_shape=(IMG_SIZE[0], IMG_SIZE[1], 3)
    )
    base_model.trainable = False  # Freeze weights
    return base_model

feat_extractor = build_feature_extractor()
print("Feature Extractor Loaded: MobileNetV2 (Frozen)")

I0000 00:00:1768940654.095701  135788 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13942 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:1e.0, compute capability: 7.5


Feature Extractor Loaded: MobileNetV2 (Frozen)


## 3. Video Processing Utils
Functions to read video frames and extract features.

In [ ]:
def extract_frames(video_path, seq_len=SEQ_LENGTH):
    """
    Extracts 'seq_len' frames from a video file evenly spaced.
    Returns: (seq_len, 224, 224, 3) preprocessed numpy array.
    """
    if not os.path.exists(video_path):
        return np.zeros((seq_len, *IMG_SIZE, 3))
    
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        return np.zeros((seq_len, *IMG_SIZE, 3))
    
    frames = []
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    
    if total_frames <= 0:
        cap.release()
        return np.zeros((seq_len, *IMG_SIZE, 3))
        
    # Uniform sampling
    indices = np.linspace(0, total_frames - 1, seq_len).astype(int)
    
    # We iterate and pick frames
    current_frame = 0
    picked_idx = 0
    
    while True:
        ret, frame = cap.read()
        if not ret:
            break
            
        if picked_idx < seq_len and current_frame == indices[picked_idx]:
            frame = cv2.resize(frame, IMG_SIZE)
            frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            frames.append(frame)
            picked_idx += 1
        
        current_frame += 1
        
        if picked_idx >= seq_len:
            break
                
    cap.release()
    
    # Pad if video was too short/corrupt
    while len(frames) < seq_len:
        frames.append(np.zeros((*IMG_SIZE, 3)))
        
    frames = np.array(frames)
    frames = preprocess_input(frames)
    return frames

In [11]:
def get_video_features(video_path):
    frames = extract_frames(video_path)
    if len(frames) == 0:
        return np.zeros(1280) # Return zero vector if empty
    
    # Extract frame-level features: (T_frames, 1280)
    frame_features = feat_extractor.predict(frames, verbose=0)
    
    # Temporal Pooling: Average frame features to get one vector per video
    # Shape: (1280,)
    video_feature = np.mean(frame_features, axis=0)
    return video_feature

## 4. Model Definition & Helpers
Sequential ESN class and data sequence utilities.

In [12]:
def create_sequential_blocks(df, is_test=False):
    """
    Groups a dataframe into sequential blocks of videos based on time_segment_id.
    Returns: List of (X_block, y_block) tuples.
    X_block: (T_videos, 1280)
    y_block: (T_videos, )
    """
    blocks = []
    
    # Ensure sorted by view/camera and time
    # Assuming 'time_segment_id' implies order. If datetime is available, sort by it.
    if 'datetimestamp_start' in df.columns:
        df['dt'] = pd.to_datetime(df['datetimestamp_start'])
        df = df.sort_values(by=['view_label', 'dt'])
    else:
        df = df.sort_values(by=['view_label', 'time_segment_id'])
    
    print(f"Grouping {len(df)} samples into sequential blocks...")
    
    grouped = df.groupby('view_label')
    
    for view_id, group in grouped:
        # Identify continuity breaks
        # We assume time_segment_id is continuous integers (0, 1, 2...) for continuous time
        # diff != 1 implies a break in the sequence
        group = group.copy()
        group['block_id'] = (group['time_segment_id'].diff() != 1).cumsum()
        
        for _, block in group.groupby('block_id'):
            # Stack features
            if 'features' not in block.columns:
                continue 
                
            X_seq = np.stack(block['features'].values)
            
            if not is_test:
                y_seq = block['label_code'].values
            else:
                y_seq = np.zeros(len(block)) # Dummy labels for test
                
            blocks.append((X_seq, y_seq, block['video_full_path'].values if is_test else None))
            
    print(f"Refined into {len(blocks)} blocks.")
    return blocks

In [13]:
def reconstruct_test_video_path(row):
    # Example Input columns: 'view_label' ('Norman Niles #1'), 'datetimestamp_start' ('2025-10-20 06:00:45')
    # Target Format: 'normanniles1/normanniles1_2025-10-20-06-00-45.mp4'
    # Or simply joining with VIDEO_DIR if folders are flattened: 'normanniles1_2025-10-20-06-00-45.mp4'
    
    # 1. Normalize Camera Name
    cam_map = {
        'Norman Niles #1': 'normanniles1',
        'Norman Niles #2': 'normanniles2',
        'Norman Niles #3': 'normanniles3',
        'Norman Niles #4': 'normanniles4'
    }
    
    cam_key = row.get('view_label', '')
    if cam_key not in cam_map:
        # Fallback if view_label is missing but implied by ID
        return None
        
    cam_id = cam_map[cam_key]
    
    # 2. Extract Date/Time
    # Assuming 'datetimestamp_start' exists in Test CSV (Standard Zindi format)
    # If not, we might need to parse 'ID' (e.g. time_segment_0... usually doesn't have date)
    # BUT: The TestInputSegments.csv provided usually has 'video_time' or 'datetimestamp_start'
    
    dt_str = row.get('datetimestamp_start', row.get('video_time', ''))
    if not dt_str:
        return None
        
    # Format: '2025-10-20 06:00:45' -> '2025-10-20-06-00-45'
    dt_formatted = str(dt_str).replace(' ', '-').replace(':', '-')
    
    # 3. Construct Path
    # Assuming folder structure 'normanniles1/filename.mp4' to match training
    filename = f"{cam_id}_{dt_formatted}.mp4"
    full_path = os.path.join(VIDEO_DIR, cam_id, filename)
    
    # Check if we need to fall back to flat directory
    if not os.path.exists(full_path):
         # Try flat in VIDEO_DIR
         full_path_flat = os.path.join(VIDEO_DIR, filename)
         # Return flat path anyway to let the loader check existence
         return full_path_flat
         
    return full_path

In [14]:
class DeepESN:
    """
    Deep Echo State Network (DeepESN) implementation.
    Consists of a stack of reservoir layers. Each layer feeds into the next.
    The final state used for prediction is the concatenation of states from all layers.
    """
    def __init__(self, input_dim=1280, n_layers=2, res_dim=1000, spectral_radius=0.9, leak_rate=0.2, ridge_alpha=1.0, random_state=42):
        self.input_dim = input_dim
        self.n_layers = n_layers
        self.res_dim = res_dim
        self.spectral_radius = spectral_radius
        self.leak_rate = leak_rate
        self.ridge_alpha = ridge_alpha
        self.random_state = random_state
        
        # Initialize Architecture
        self.layers = [] # List of dicts {'W_in', 'W_res'}
        rng = np.random.RandomState(self.random_state)
        
        for i in range(n_layers):
            # Input dimension for layer i: 
            # Layer 0 takes original input (input_dim)
            # Layer >0 takes state of previous layer (res_dim)
            curr_input_dim = input_dim if i == 0 else res_dim
            
            # Input weights
            W_in = rng.uniform(-1, 1, (res_dim, curr_input_dim))
            
            # Reservoir weights (Sparse)
            W_res = rng.uniform(-1, 1, (res_dim, res_dim))
            mask = rng.rand(res_dim, res_dim) > 0.95
            W_res[mask] = 0
            
            # Spectral Radius Scaling
            try:
                eigenvalues = np.linalg.eigvals(W_res)
                max_eig = np.max(np.abs(eigenvalues))
                if max_eig > 0:
                    W_res *= (self.spectral_radius / max_eig)
            except:
                W_res *= 0.9 # Fallback
                
            self.layers.append({
                'W_in': W_in,
                'W_res': W_res
            })
            
        self.readout = Ridge(alpha=self.ridge_alpha)
        
    def get_states_sequence(self, input_seq):
        """
        Processes a sequence (T, input_dim) through the Deep ESN stack.
        Returns: (T, n_layers * res_dim) - Concatenated states of all layers
        """
        T = input_seq.shape[0]
        
        # We need to store full sequence of states for each layer to feed to the next
        # layer_states: prediction input for next layer
        prev_layer_seq = input_seq # Start with actual input
        
        all_layers_collected_states = [] # To be concatenated for readout
        
        for i, layer in enumerate(self.layers):
            W_in = layer['W_in']
            W_res = layer['W_res']
            
            current_layer_states = np.zeros((T, self.res_dim))
            x = np.zeros(self.res_dim)
            
            for t in range(T):
                u = prev_layer_seq[t]
                
                # Standard ESN Equestion: x(t) = (1-a)x(t-1) + a*tanh(Win*u + Wres*x(t-1))
                pre = np.dot(W_in, u) + np.dot(W_res, x)
                update = np.tanh(pre)
                x = (1 - self.leak_rate) * x + self.leak_rate * update
                
                current_layer_states[t] = x
            
            all_layers_collected_states.append(current_layer_states)
            prev_layer_seq = current_layer_states # Output of this layer is input to next
            
        # Concatenate all layers: (T, n_layers * res_dim)
        final_states = np.hstack(all_layers_collected_states)
        return final_states

    def fit(self, blocks, compute_metrics=True):
        """
        Trains the execution readout on sequential blocks.
        blocks: List of (X_seq, y_seq, ...)
        """
        all_states = []
        all_targets = []
        
        print(f"DeepESN: Training on {len(blocks)} blocks (Layers={self.n_layers}, ResDim={self.res_dim})...")
        
        for idx, (X_seq, y_seq, _) in enumerate(blocks):
            states_seq = self.get_states_sequence(X_seq)
            all_states.append(states_seq)
            all_targets.append(y_seq)
            
        # Stack
        X_train_res = np.vstack(all_states)
        y_train_flat = np.concatenate(all_targets)
        
        self.readout.fit(X_train_res, y_train_flat)
        
        if compute_metrics:
            y_pred = self.readout.predict(X_train_res)
            y_pred_class = np.round(np.clip(y_pred, 0, 3)).astype(int)
            acc = accuracy_score(y_train_flat, y_pred_class)
            f1 = f1_score(y_train_flat, y_pred_class, average='macro')
            print(f"[TRAIN] DeepESN Accuracy: {acc:.4f} | F1-Macro: {f1:.4f}")
            
    def predict(self, blocks):
        all_preds = []
        for X_seq, _, _ in blocks:
            states_seq = self.get_states_sequence(X_seq)
            preds_seq = self.readout.predict(states_seq)
            all_preds.append(preds_seq)
        return all_preds

## 5. Training Pipeline
Loads training data, extracts features, trains ESN, and reports validation metrics.

In [15]:
# --- 5. Train & Validation Execution ---

# 1. Load Training Data
print("Loading Training Data...")
df_train = pd.read_csv(TRAIN_CSV)
df_train['video_full_path'] = df_train['videos'].apply(extract_camera_path)

# Map labels
congestion_map = {'free flowing': 0, 'light delay': 1, 'moderate delay': 2, 'heavy delay': 3}
df_train['label_code'] = df_train['congestion_enter_rating'].map(congestion_map).fillna(0).astype(int)

# 2. Extract Features
print("Extracting features for ALL Training videos...")
feats_list = []
total = len(df_train)
for idx, row in df_train.iterrows():
    if idx % 100 == 0: print(f"{idx}/{total}")
    # Note: get_video_features now handles the frame extraction + mean pooling
    f = get_video_features(row['video_full_path'])
    feats_list.append(f)

df_train['features'] = feats_list

# 3. Create Blocks
print("Creating Sequential Blocks...")
train_blocks = create_sequential_blocks(df_train, is_test=False)

# 4. Train/Val Split (Time-based: Last 20% of blocks)
# This respects the temporal order we just established
n_val = int(len(train_blocks) * 0.2)
train_blocks_split = train_blocks[:-n_val]
val_blocks_split = train_blocks[-n_val:]

print(f"Total Blocks: {len(train_blocks)}")
print(f"Training on {len(train_blocks_split)} blocks, Validating on {len(val_blocks_split)} blocks.")

# 5. Initialize & Train Deep ESN (Updated)
print("Initializing and Training DeepESN...")
# Defaulting to n_layers=2 and res_dim=1500 as per tuning insights (Deep > Shallow)
esn = DeepESN(input_dim=1280, n_layers=2, res_dim=1500, spectral_radius=0.95, leak_rate=0.2, ridge_alpha=1.0) 
esn.fit(train_blocks_split, compute_metrics=True) 

# 6. Validation
print("Running Validation...")
val_preds_list = esn.predict(val_blocks_split)

# Flatten for metrics
y_val_true = []
y_val_pred = []

for i, p_seq in enumerate(val_preds_list):
    _, t_seq, _ = val_blocks_split[i]
    y_val_true.extend(t_seq)
    y_val_pred.extend(p_seq)

y_val_pred_class = np.round(np.clip(y_val_pred, 0, 3)).astype(int)

print("-" * 30)
print("FINAL VALIDATION RESULTS:")
print(f"Accuracy: {accuracy_score(y_val_true, y_val_pred_class):.4f}")
print(f"F1-Macro: {f1_score(y_val_true, y_val_pred_class, average='macro'):.4f}")
print("-" * 30)

Loading Training Data...
Extracting features for ALL Training videos...
0/3000


NameError: name 'extract_frames' is not defined

## 6. Inference on Test Set
Reconstructs test video paths, processes sequences, and generates submission.

In [94]:
# --- 5. Inference (Test Blocks) ---

# 1. Load Test Data
df_test = pd.read_csv(TEST_CSV)
print(f"Test Set Size: {len(df_test)}")

# --- FIX: Resolve ID Mismatch ---
# The submission expects an 'ID' column, but test has 'ID_enter' and 'ID_exit'.
# Based on sample submission, we are predicting 'congestion_enter_rating'.
if 'ID' not in df_test.columns:
    if 'ID_enter' in df_test.columns:
        df_test['ID'] = df_test['ID_enter']
    else:
        # Fallback or Error
        print("WARNING: Could not find 'ID' or 'ID_enter' column. Submission will fail.")

# 2. Reconstruct Paths
print("Reconstructing Test Video Paths...")
df_test['video_full_path'] = df_test.apply(reconstruct_test_video_path, axis=1)

# 3. Extract Features (Robust Loop)
print("Extracting Test Features...")
test_feats = []
missing_count = 0
total_test = len(df_test)

# OPTIONAL: Limit for quick testing (Comment out for full run)
# df_test = df_test.iloc[:100] 

for idx, row in df_test.iterrows():
    if idx % 100 == 0: print(f"{idx}/{total_test}")
    path = row['video_full_path']
    
    if path and os.path.exists(path):
        try:
            f = get_video_features(path)
            test_feats.append(f)
        except Exception as e:
            # print(f"Error reading {path}: {e}")
            test_feats.append(np.zeros(1280))
            missing_count += 1
    else:
        # Handle missing video
        test_feats.append(np.zeros(1280))
        missing_count += 1
        
print(f"Test Extraction Complete. Missing/Error Videos: {missing_count}/{total_test}")
df_test['features'] = test_feats

# 4. Sequential Prediction
test_blocks = create_sequential_blocks(df_test, is_test=True)
print(f"Predicting on {len(test_blocks)} test blocks...")
test_preds_list = esn.predict(test_blocks)

# 5. Map Predictions to ID
pred_map = {}
for i, p_seq in enumerate(test_preds_list):
    _, _, paths_seq = test_blocks[i]
    for j, path in enumerate(paths_seq):
        # Clamp predictions to valid range [0, 3]
        pred_val = np.round(np.clip(p_seq[j], 0, 3)).astype(int)
        if path is not None:
            pred_map[path] = pred_val

df_test['pred_code'] = df_test['video_full_path'].map(pred_map).fillna(0).astype(int)

# 6. Generate Submission
congestion_inv_map = {0: 'free flowing', 1: 'light delay', 2: 'moderate delay', 3: 'heavy delay'}
submission = df_test[['ID']].copy()
submission['Target'] = df_test['pred_code'].map(congestion_inv_map)

submission.to_csv('submission_esn.csv', index=False)
print("Submission saved to 'submission_esn.csv'")
print(submission.head())

Test Set Size: 2640
Reconstructing Test Video Paths...
Extracting Test Features...
0/2640
100/2640
200/2640
300/2640
400/2640
500/2640
600/2640
700/2640
800/2640
900/2640
1000/2640
1100/2640
1200/2640
1300/2640
1400/2640
1500/2640
1600/2640
1700/2640
1800/2640
1900/2640
2000/2640
2100/2640
2200/2640
2300/2640
2400/2640
2500/2640
2600/2640
Test Extraction Complete. Missing Videos: 2335/2640
Validation Metrics (Recap):
Validation F1-Macro: 0.0658
Grouping 2640 samples into sequential blocks...
Refined into 185 blocks.


KeyError: "None of [Index(['ID'], dtype='object')] are in the [columns]"